In [3]:


import os
import math
import time
import json
import random
import warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from scipy.optimize import minimize
from tqdm import tqdm
from dataclasses import dataclass
from typing import List, Tuple, Dict

warnings.filterwarnings("ignore")


# ══════════════════════════════════════════════════════════════
#  SECTION 1: CONFIGURATION
# ══════════════════════════════════════════════════════════════

@dataclass
class Config:
    expr_col: str = "expression_prefix_masked"
    max_vars: int = 3
    max_expr_len: int = 50
    n_subsample: int = 200
    noise_scale: float = 0.03

    d_model: int = 128
    n_heads: int = 4
    dec_layers: int = 4
    dropout: float = 0.3

    beam_width: int = 10
    top_k: int = 5
    max_residual_steps: int = 5
    residual_threshold: float = 1e-6

    batch_size: int = 128
    lr: float = 3e-4
    weight_decay: float = 1e-4
    warmup: int = 1500
    epochs: int = 60
    patience: int = 15
    grad_clip: float = 1.0
    seed: int = 42
    amp: bool = True

    bfgs_restarts: int = 5
    bfgs_maxiter: int = 500
    ckpt_dir: str = "./checkpoints"
    log_dir: str = "./logs"


# ══════════════════════════════════════════════════════════════
#  SECTION 2: DATA-DRIVEN VOCABULARY
#  Scans CSV → collects tokens → infers arities by trial-parsing
# ══════════════════════════════════════════════════════════════

def _is_var(t):
    """Detect variable tokens like x_1, x_2, x_3."""
    s = t.lower().replace("_", "")
    return len(s) >= 2 and s[0] == "x" and s[1:].isdigit()


def _is_num(t):
    """Detect numeric literals."""
    try:
        float(t)
        return True
    except:
        return False


def _is_const(t):
    """Detect constant placeholders like <C_0>, <C_1>, etc."""
    return t.startswith("<C_")


def _try_parse(toks, pos, ar_map):
    """Recursive prefix parser to test arity assignments."""
    if pos >= len(toks):
        return pos, False
    a = ar_map.get(toks[pos], 0)
    pos += 1
    for _ in range(a):
        pos, ok = _try_parse(toks, pos, ar_map)
        if not ok:
            return pos, False
    return pos, True


def build_vocab(df, expr_col):
    """
    Build vocabulary entirely from data.
    Scans expressions, collects tokens, infers arities.
    """
    all_toks = set()
    exprs = []
    for e in df[expr_col].dropna().astype(str):
        e = e.strip()
        if e and e != "nan":
            all_toks.update(e.split())
            exprs.append(e)

    # Classify tokens
    variables = sorted(t for t in all_toks if _is_var(t))
    numbers = sorted(t for t in all_toks if _is_num(t))
    operators = sorted(t for t in all_toks if not _is_var(t) and not _is_num(t))

    # Base arities: variables and numbers are terminals
    ar_map = {}
    for v in variables:
        ar_map[v] = 0
    for n in numbers:
        ar_map[n] = 0

    # Infer arity for each operator by trial parsing
    print(f"  Inferring arities for {len(operators)} tokens...")
    sample = exprs[:3000]
    for tok in operators:
        best_a = 0
        best_s = -1
        for try_a in [0, 1, 2]:
            ar_map[tok] = try_a
            score = 0
            for e in sample:
                if tok not in e.split():
                    continue
                end, ok = _try_parse(e.split(), 0, ar_map)
                if ok and end == len(e.split()):
                    score += 1
            if score > best_s:
                best_s = score
                best_a = try_a
        ar_map[tok] = best_a

    # Categorize
    binary = sorted(t for t in operators if ar_map[t] == 2)
    unary = sorted(t for t in operators if ar_map[t] == 1)
    terms = sorted(t for t in operators if ar_map[t] == 0)

    # Build token list
    tlist = ["<PAD>", "<SOS>", "<EOS>"] + binary + unary + variables + terms
    ar_map["<PAD>"] = 0
    ar_map["<SOS>"] = 0
    ar_map["<EOS>"] = 0

    # Build vocab dict
    vocab = {t: i for i, t in enumerate(tlist)}
    i2t = {i: t for t, i in vocab.items()}

    # Build arity lookup by token ID
    arity = {}
    for t in tlist:
        arity[vocab[t]] = ar_map.get(t, 0)

    # Precompute ID sets for grammar mask
    op_ids = {vocab[t] for t in binary + unary}
    term_ids = set()
    for i in range(len(tlist)):
        if i not in op_ids and i not in {vocab["<PAD>"], vocab["<SOS>"], vocab["<EOS>"]}:
            term_ids.add(i)

    # Verify parsing
    n_ok = 0
    for e in exprs[:5000]:
        end, ok = _try_parse(e.split(), 0, ar_map)
        if ok and end == len(e.split()):
            n_ok += 1

    print(f"  Vocab: {len(vocab)}  binary:{binary}  unary:{unary}")
    print(f"  vars:{variables}")
    print(f"  terms:{terms[:10]}{'...' if len(terms) > 10 else ''}")
    print(f"  Parse verify: {n_ok}/{min(len(exprs), 5000)}")

    meta = {
        "arity": arity,
        "op_ids": op_ids,
        "term_ids": term_ids,
        "ar_map": ar_map,
    }
    return vocab, i2t, meta


def tokenize(expr, vocab):
    """Convert prefix expression string → token IDs."""
    if not isinstance(expr, str) or not expr.strip():
        return None
    ids = [vocab["<SOS>"]]
    for t in expr.strip().split():
        if t in vocab:
            ids.append(vocab[t])
        elif _is_num(t):
            # If there's a NUM token in vocab, use it; otherwise skip
            if "NUM" in vocab:
                ids.append(vocab["NUM"])
            else:
                return None
        else:
            return None
    ids.append(vocab["<EOS>"])
    return ids


def valid_prefix(ids, vocab, meta):
    """Check if token ID sequence is a structurally valid prefix expression."""
    skip = {vocab["<PAD>"], vocab["<SOS>"], vocab["<EOS>"]}
    cl = [i for i in ids if i not in skip]
    if not cl:
        return False
    ar = 1
    for i in cl:
        if ar <= 0:
            return False
        ar = ar - 1 + meta["arity"].get(i, 0)
    return ar == 0


# ══════════════════════════════════════════════════════════════
#  SECTION 3: DATA LOADING
# ══════════════════════════════════════════════════════════════

def get_valid_indices(npz_path, total=50000):
    """Scan NPZ for equations with valid X data."""
    data = np.load(npz_path)
    valid = []
    print("Scanning NPZ...")
    for i in range(total):
        try:
            if f"X_{i}" in data and data[f"X_{i}"].shape[0] > 0:
                valid.append(i)
        except:
            continue
    print(f"Found {len(valid)} valid / {total}")
    return valid


def load_into_ram(npz_path, valid_idx, max_vars):
    """Load all valid data into RAM arrays."""
    print("Loading into RAM...")
    fd = np.load(npz_path)
    pts = np.zeros((len(valid_idx), 500, max_vars + 1), dtype=np.float32)
    lens = np.zeros(len(valid_idx), dtype=np.int32)
    kept = []
    k = 0

    for ri in tqdm(valid_idx, desc="Loading"):
        try:
            xp = np.array(fd[f"X_{ri}"], dtype=np.float32)
            yk = f"Y_{ri}" if f"Y_{ri}" in fd else f"y_{ri}"
            yp = np.array(fd[yk], dtype=np.float32).ravel()
        except:
            continue

        if xp.ndim == 1:
            xp = xp.reshape(-1, 1)

        np_ = min(xp.shape[0], len(yp), 500)
        nv = min(xp.shape[1], max_vars)
        if np_ < 4:
            continue

        x = xp[:np_, :nv].copy()
        y = yp[:np_].copy()

        # Filter NaN/Inf
        ok = np.isfinite(x).all(1) & np.isfinite(y)
        x = x[ok]
        y = y[ok]

        if len(y) < 4:
            continue
        if y.std() < 1e-12:
            continue

        np_ = len(y)
        pts[k, :np_, :nv] = x
        pts[k, :np_, max_vars] = y
        lens[k] = np_
        kept.append(ri)
        k += 1

    pts = pts[:k]
    lens = lens[:k]
    print(f"  Loaded {k}, shape {pts.shape}")
    return pts, lens, kept


# ══════════════════════════════════════════════════════════════
#  SECTION 4: STATISTICAL FEATURES (NeSymReS-Inspired)
#  Hand-crafted: slopes, curvature, FFT, correlations.
#  No learnable params — cannot overfit.
# ══════════════════════════════════════════════════════════════

def n_sf(mv):
    """Number of statistical features for mv input variables."""
    return 4 * (mv + 1) + 5 * mv


def compute_stats(dp, length, mv):
    """Extract fixed statistical features from a data cloud."""
    d = dp[:length]
    X = d[:, :mv]
    y = d[:, mv]
    f = []

    # Per-column: mean, std, min, max
    for c in range(mv + 1):
        v = d[:, c]
        f.append(np.mean(v))
        f.append(np.std(v) + 1e-10)
        f.append(np.min(v))
        f.append(np.max(v))

    # Per-variable: slope, curvature, corr, fft, sign changes
    for vi in range(mv):
        xi = X[:, vi]
        idx = np.argsort(xi)
        xs = xi[idx]
        ys = y[idx]

        dx = np.diff(xs)
        dx = np.where(np.abs(dx) < 1e-10, 1e-10, dx)
        sl = np.diff(ys) / dx

        # Mean absolute slope
        f.append(np.mean(np.abs(sl)))

        # Mean absolute curvature
        if len(sl) > 1:
            f.append(np.mean(np.abs(np.diff(sl))))
        else:
            f.append(0.0)

        # Correlation
        corr = np.corrcoef(xi, y)[0, 1]
        if np.isfinite(corr):
            f.append(corr)
        else:
            f.append(0.0)

        # FFT dominant frequency
        if len(ys) > 10:
            yd = ys - np.linspace(ys[0], ys[-1], len(ys))
            fm = np.abs(np.fft.rfft(yd))
            if len(fm) > 1:
                f.append((np.argmax(fm[1:]) + 1) / len(ys))
            else:
                f.append(0.0)
        else:
            f.append(0.0)

        # Sign change rate
        f.append(np.sum(np.diff(np.sign(ys)) != 0) / max(len(ys) - 1, 1))

    out = np.array(f, dtype=np.float32)
    out = np.nan_to_num(out, nan=0.0, posinf=100.0, neginf=-100.0)
    out = np.clip(out, -100.0, 100.0)
    return out


def precompute_stats(pts, lens, mv):
    """Compute stats for all equations once before training."""
    nsf = n_sf(mv)
    sf = np.zeros((len(lens), nsf), dtype=np.float32)
    print("Precomputing stats...")
    for i in tqdm(range(len(lens)), desc="Stats"):
        sf[i] = compute_stats(pts[i], lens[i], mv)
    return sf


# ══════════════════════════════════════════════════════════════
# ══════════════════════════════════════════════════════════════

class SRDataset(Dataset):
    def __init__(self, pts, lens, stats, tok_padded, cfg, augment=True):
        self.pts = pts
        self.lens = lens
        self.stats = stats
        self.toks = tok_padded
        self.cfg = cfg
        self.aug = augment
        self.ns = cfg.n_subsample
        print(f"  Dataset: {len(self.toks)}, aug={augment}")

    def __len__(self):
        return self.pts.shape[0]

    def __getitem__(self, idx):
        pts = self.pts[idx]       # (500, mv+1)
        n = self.lens[idx].item()
        sf = self.stats[idx]      # (nsf,)
        toks = self.toks[idx]     # (max_expr_len,)

        if self.aug:
            # Subsample
            ns = min(self.ns, n)
            sel = torch.randperm(n)[:ns]
            dp = pts[sel]

            # Add noise
            std = dp.std(0, keepdim=True).clamp(min=1e-8)
            dp = dp + torch.randn_like(dp) * self.cfg.noise_scale * std
        else:
            dp = pts[:min(200, n)]

        # Normalize
        mu = dp.mean(0, keepdim=True)
        sig = dp.std(0, keepdim=True).clamp(min=1e-8)
        dp = ((dp - mu) / sig).clamp(-10.0, 10.0)

        # Pad to 500
        a = dp.shape[0]
        if a < 500:
            pad = torch.zeros(500 - a, dp.shape[1])
            dp = torch.cat([dp, pad], 0)
            mk = torch.zeros(500)
            mk[:a] = 1.0
        else:
            dp = dp[:500]
            mk = torch.ones(500)

        return dp, mk, sf, toks, toks.shape[0]


# ══════════════════════════════════════════════════════════════
#  SECTION 6: MODEL
#  Encoder: DeepSets — per-point MLP + mean pool + stat features
#  Decoder: Transformer with grammar mask at inference only
# ══════════════════════════════════════════════════════════════

class Encoder(nn.Module):
    """DeepSets encoder (Zaheer et al., 2017) + statistical features."""

    def __init__(self, cfg):
        super().__init__()
        d = cfg.d_model
        inp = cfg.max_vars + 1

        self.mlp = nn.Sequential(
            nn.Linear(inp, d),
            nn.LayerNorm(d),
            nn.GELU(),
            nn.Dropout(cfg.dropout),
            nn.Linear(d, d * 2),
            nn.LayerNorm(d * 2),
            nn.GELU(),
            nn.Dropout(cfg.dropout),
            nn.Linear(d * 2, d),
            nn.LayerNorm(d),
            nn.GELU(),
            nn.Dropout(cfg.dropout),
        )

        self.sp = nn.Sequential(
            nn.Linear(n_sf(cfg.max_vars), d),
            nn.GELU(),
            nn.Dropout(cfg.dropout),
            nn.Linear(d, d),
        )

        self.fuse = nn.Sequential(
            nn.Linear(d * 2, d),
            nn.GELU(),
            nn.Dropout(cfg.dropout),
            nn.Linear(d, d),
        )

    def forward(self, pts, mask, sf):
        # Per-point MLP
        h = self.mlp(pts)                    # (B, 500, d)

        # Masked mean pool
        m = mask.unsqueeze(-1)               # (B, 500, 1)
        h = (h * m).sum(1) / (m.sum(1) + 1e-8)  # (B, d)

        # Fuse with stats
        s = self.sp(sf)                      # (B, d)
        ctx = self.fuse(torch.cat([h, s], -1))  # (B, d)
        return ctx.unsqueeze(1)              # (B, 1, d)


class Decoder(nn.Module):
    """Transformer decoder (Lample & Charton, 2019) with grammar mask."""

    def __init__(self, cfg, vocab, meta):
        super().__init__()
        self.cfg = cfg
        d = cfg.d_model
        vs = len(vocab)

        self.emb = nn.Embedding(vs, d, padding_idx=vocab["<PAD>"])
        self.pe = nn.Embedding(cfg.max_expr_len, d)

        layer = nn.TransformerDecoderLayer(
            d_model=d,
            nhead=cfg.n_heads,
            dim_feedforward=d * 4,
            dropout=cfg.dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )
        self.dec = nn.TransformerDecoder(layer, cfg.dec_layers)
        self.proj = nn.Linear(d, vs)

        self.pad = vocab["<PAD>"]
        self.sos = vocab["<SOS>"]
        self.eos = vocab["<EOS>"]
        self.op_ids = meta["op_ids"]
        self.term_ids = meta["term_ids"]
        self.arity = meta["arity"]

    def forward(self, ctx, tgt, grammar=False):
        B, S = tgt.shape

        # Embed tokens + positional encoding
        pos = torch.arange(S, device=tgt.device).unsqueeze(0)
        x = self.emb(tgt) + self.pe(pos)

        # Causal mask + padding mask
        cm = nn.Transformer.generate_square_subsequent_mask(S, device=tgt.device)
        pad_mask = (tgt == self.pad)

        # Decode
        out = self.dec(x, ctx, tgt_mask=cm, tgt_key_padding_mask=pad_mask)
        lo = self.proj(out)

        # Grammar mask (only during beam search)
        if grammar:
            lo = self._gram(lo, tgt)
        return lo

    def _gram(self, lo, tgt):
        """Grammar mask — beam search only, never during training."""
        B, S, V = lo.shape
        for b in range(B):
            ar = 1
            for t in range(1, S):
                ok = torch.zeros(V, dtype=torch.bool, device=lo.device)
                left = self.cfg.max_expr_len - t - 1

                if ar == 0:
                    ok[self.eos] = True
                elif left <= ar:
                    for i in self.term_ids:
                        ok[i] = True
                else:
                    for i in self.op_ids | self.term_ids:
                        ok[i] = True

                lo[b, t, ~ok] = float("-inf")

                tk = tgt[b, t].item()
                if tk in (self.pad, self.eos):
                    break
                ar = ar - 1 + self.arity.get(tk, 0)
        return lo


class SymbolicRegressionModel(nn.Module):
    def __init__(self, cfg, vocab, meta):
        super().__init__()
        self.enc = Encoder(cfg)
        self.dec = Decoder(cfg, vocab, meta)

    def forward(self, pts, mask, sf, tgt, grammar=False):
        ctx = self.enc(pts, mask, sf)
        return self.dec(ctx, tgt, grammar)

    def n_params(self):
        return sum(p.numel() for p in self.parameters() if p.requires_grad)


# ══════════════════════════════════════════════════════════════
#  SECTION 7: BEAM SEARCH (grammar-constrained)
# ══════════════════════════════════════════════════════════════

@torch.no_grad()
def beam_search(model, pts, mk, sf, vocab, meta, cfg, device):
    """Grammar-constrained beam search. Returns [(token_ids, log_prob), ...]."""
    model.eval()
    dec = model.dec
    ctx = model.enc(pts.to(device), mk.to(device), sf.to(device))

    beams = [{"t": [vocab["<SOS>"]], "lp": 0.0, "ar": 1, "done": False}]

    for step in range(1, cfg.max_expr_len):
        act = [b for b in beams if not b["done"]]
        done = [b for b in beams if b["done"]]
        if not act:
            break

        cs = []
        for b in act:
            t = torch.tensor([b["t"]], dtype=torch.long, device=device)
            pos = torch.arange(t.size(1), device=device).unsqueeze(0)
            x = dec.emb(t) + dec.pe(pos)
            causal = nn.Transformer.generate_square_subsequent_mask(
                t.size(1), device=device
            )
            out = dec.dec(x, ctx, tgt_mask=causal)
            lo = dec.proj(out[:, -1, :])

            # Grammar mask
            ok = torch.zeros(len(vocab), dtype=torch.bool, device=device)
            left = cfg.max_expr_len - step - 1

            if b["ar"] == 0:
                ok[dec.eos] = True
            elif left <= b["ar"]:
                for i in dec.term_ids:
                    ok[i] = True
            else:
                for i in dec.op_ids | dec.term_ids:
                    ok[i] = True

            lo[0, ~ok] = float("-inf")
            lo[0, dec.pad] = float("-inf")
            lo[0, dec.sos] = float("-inf")

            lps = F.log_softmax(lo, -1)
            k = min(cfg.beam_width, ok.sum().item())
            if k == 0:
                continue

            tl, ti = torch.topk(lps[0], k)

            for j in range(ti.size(0)):
                tid = ti[j].item()
                tlp = tl[j].item()
                if tlp == float("-inf"):
                    continue

                nt = b["t"] + [tid]
                nlp = b["lp"] + tlp

                if tid == dec.eos:
                    cs.append({"t": nt, "lp": nlp, "ar": 0, "done": True})
                else:
                    new_ar = b["ar"] - 1 + meta["arity"].get(tid, 0)
                    cs.append({"t": nt, "lp": nlp, "ar": new_ar, "done": False})

        cs.extend(done)
        cs.sort(key=lambda b: b["lp"] / (len(b["t"]) ** 0.7), reverse=True)
        beams = cs[:cfg.beam_width]

        if all(b["done"] for b in beams):
            break

    res = sorted(
        [b for b in beams if b["done"]],
        key=lambda b: b["lp"],
        reverse=True,
    )
    if not res:
        res = beams[:1]
    return [(b["t"], b["lp"]) for b in res[:cfg.top_k]]


# ══════════════════════════════════════════════════════════════
#  SECTION 8: EXPRESSION EVALUATION + BFGS
# ══════════════════════════════════════════════════════════════

OPS = {
    "add": lambda a, b: a + b,
    "sub": lambda a, b: a - b,
    "mul": lambda a, b: a * b,
    "div": lambda a, b: np.where(np.abs(b) < 1e-10, 0.0, a / b),
    "pow": lambda a, b: np.clip(
        np.power(np.abs(a) + 1e-10, np.clip(b, -5, 5)), -1e10, 1e10
    ),
    "sin": np.sin,
    "cos": np.cos,
    "exp": lambda a: np.clip(np.exp(np.clip(a, -20, 20)), -1e10, 1e10),
    "log": lambda a: np.log(np.abs(a) + 1e-10),
    "ln": lambda a: np.log(np.abs(a) + 1e-10),
    "sqrt": lambda a: np.sqrt(np.abs(a)),
    "abs": np.abs,
    "neg": lambda a: -a,
    "tan": lambda a: np.where(np.abs(np.cos(a)) < 1e-10, 0.0, np.tan(a)),
    "inv": lambda a: np.where(np.abs(a) < 1e-10, 0.0, 1.0 / a),
    "square": lambda a: a * a,
    "cube": lambda a: a * a * a,
    "asin": np.arcsin,
    "acos": np.arccos,
    "atan": np.arctan,
    "sinh": np.sinh,
    "cosh": np.cosh,
    "tanh": np.tanh,
    "sign": np.sign,
    "log2": lambda a: np.log2(np.abs(a) + 1e-10),
    "log10": lambda a: np.log10(np.abs(a) + 1e-10),
    "atan2": np.arctan2,
}


def eval_prefix(toks, X, consts, ar_map):
    """Evaluate a prefix expression on data X with given constants."""
    st = {"p": 0, "c": 0}

    def go():
        if st["p"] >= len(toks):
            return np.zeros(X.shape[0])

        t = toks[st["p"]]
        st["p"] += 1
        a = ar_map.get(t, 0)

        # Binary operator
        if a == 2 and t in OPS:
            left = go()
            right = go()
            return OPS[t](left, right)

        # Unary operator
        if a == 1 and t in OPS:
            child = go()
            return OPS[t](child)

        # Constant placeholder: <C_0>, <C_1>, etc.
        if _is_const(t):
            i = st["c"]
            st["c"] += 1
            if i < len(consts):
                return np.full(X.shape[0], consts[i])
            return np.full(X.shape[0], 0.0)

        # Variable: x_1, x_2, x_3
        if _is_var(t):
            vi = int(t.lower().replace("_", "")[1:]) - 1
            if vi < X.shape[1]:
                return X[:, vi]
            return np.zeros(X.shape[0])

        # Numeric literal
        if _is_num(t):
            return np.full(X.shape[0], float(t))

        # Unknown token
        return np.zeros(X.shape[0])

    try:
        r = go()
        r = np.nan_to_num(r, nan=0.0, posinf=1e10, neginf=-1e10)
        return np.clip(r, -1e10, 1e10)
    except:
        return np.full(X.shape[0], np.inf)


def bfgs_fit(toks, X, y, cfg, ar_map):
    """Optimize constants in expression via L-BFGS-B."""
    # Count constant placeholders
    nc = sum(1 for t in toks if _is_const(t))

    if nc == 0:
        yp = eval_prefix(toks, X, np.array([]), ar_map)
        mse = float(np.mean((yp - y) ** 2))
        return " ".join(toks), mse

    best_mse = float("inf")
    best_c = np.zeros(nc)

    def obj(c):
        yp = eval_prefix(toks, X, c, ar_map)
        m = np.mean((yp - y) ** 2)
        if np.isfinite(m):
            return m
        return 1e20

    for r in range(cfg.bfgs_restarts):
        if r == 0:
            x0 = np.zeros(nc)
        else:
            x0 = np.random.randn(nc) * 2

        try:
            res = minimize(
                obj, x0,
                method="L-BFGS-B",
                options={"maxiter": cfg.bfgs_maxiter},
            )
            if res.fun < best_mse:
                best_mse = res.fun
                best_c = res.x
        except:
            pass

    # Substitute constants back into expression
    out = []
    ci = 0
    for t in toks:
        if _is_const(t):
            out.append(f"{best_c[ci]:.6g}")
            ci += 1
        else:
            out.append(t)

    return " ".join(out), best_mse


# ══════════════════════════════════════════════════════════════
#  SECTION 9: RESIDUAL SYMBOLIC REGRESSION (Novel Contribution)
#
#  Step 1: Predict E_1 from (X, Y)
#  Step 2: R_1 = Y - E_1(X), predict E_2 from (X, R_1)
#  Step 3: R_2 = Y - (E_1+E_2)(X), predict E_3 from (X, R_2)
#  Each step finds ONE simple pattern.
# ══════════════════════════════════════════════════════════════

def residual_predict(model, X_raw, y_raw, vocab, i2t, meta, cfg, device, verbose=True):
    """Iterative residual symbolic regression."""
    sub_exprs = []
    residual = y_raw.copy()
    X = X_raw.copy()

    # Pad to max_vars
    if X.shape[1] < cfg.max_vars:
        pad = np.zeros((X.shape[0], cfg.max_vars - X.shape[1]))
        X = np.hstack([X, pad])

    for step in range(cfg.max_residual_steps):
        cur_mse = np.mean(residual ** 2)

        # Check if we're done
        if cur_mse < cfg.residual_threshold:
            if verbose:
                print(f"  Step {step}: MSE={cur_mse:.2e} < threshold. Done.")
            break

        # Prepare input
        dp = np.column_stack([X, residual]).astype(np.float32)
        sf = compute_stats(dp, len(dp), cfg.max_vars)

        # Normalize
        mu = dp.mean(0, keepdims=True)
        sig = dp.std(0, keepdims=True) + 1e-8
        dn = ((dp - mu) / sig).astype(np.float32)
        dn = np.clip(dn, -10.0, 10.0)

        # Pad to 500
        n = dn.shape[0]
        if n < 500:
            pad = np.zeros((500 - n, dn.shape[1]), dtype=np.float32)
            dn = np.vstack([dn, pad])
            mk = np.zeros(500, dtype=np.float32)
            mk[:n] = 1.0
        else:
            dn = dn[:500]
            mk = np.ones(500, dtype=np.float32)

        # Beam search
        dp_t = torch.tensor(dn).unsqueeze(0)
        mk_t = torch.tensor(mk).unsqueeze(0)
        sf_t = torch.tensor(sf).unsqueeze(0)
        cands = beam_search(model, dp_t, mk_t, sf_t, vocab, meta, cfg, device)

        # Find best candidate via BFGS
        best_expr = None
        best_mse_c = float("inf")
        best_toks = None

        for tok_ids, lp in cands:
            strs = []
            for i in tok_ids:
                if i in i2t and i2t[i] not in {"<PAD>", "<SOS>", "<EOS>"}:
                    strs.append(i2t[i])
            if not strs:
                continue
            try:
                expr, mse = bfgs_fit(strs, X, residual, cfg, meta["ar_map"])
                if mse < best_mse_c:
                    best_mse_c = mse
                    best_expr = expr
                    best_toks = strs
            except:
                continue

        if best_toks is None:
            if verbose:
                print(f"  Step {step}: No valid candidate. Stopping.")
            break

        # Evaluate fitted expression and update residual
        nc = sum(1 for t in best_toks if _is_const(t))
        bc = np.zeros(nc)
        if nc > 0:
            try:
                def obj_fn(c):
                    yp = eval_prefix(best_toks, X, c, meta["ar_map"])
                    return np.mean((yp - residual) ** 2)
                res = minimize(obj_fn, np.zeros(nc), method="L-BFGS-B")
                bc = res.x
            except:
                pass

        yp = eval_prefix(best_toks, X, bc, meta["ar_map"])
        new_res = residual - yp
        new_mse = np.mean(new_res ** 2)

        # Only keep if it actually improves
        if new_mse >= cur_mse * 0.95:
            if verbose:
                print(f"  Step {step}: No improvement. Stopping.")
            break

        residual = new_res
        sub_exprs.append(best_expr)

        if verbose:
            print(f"  Step {step}: E_{step+1} = {best_expr}")
            print(f"           MSE: {cur_mse:.2e} → {new_mse:.2e}")

    # Combine
    if sub_exprs:
        combined = " + ".join(sub_exprs)
    else:
        combined = "<C_0>"

    # Metrics
    ss_tot = np.sum((y_raw - np.mean(y_raw)) ** 2)
    ss_res = np.sum(residual ** 2)
    r2 = 1.0 - ss_res / max(ss_tot, 1e-10)

    return {
        "sub_expressions": sub_exprs,
        "combined": combined,
        "n_steps": len(sub_exprs),
        "mse": float(np.mean(residual ** 2)),
        "r2": float(r2),
    }


# ══════════════════════════════════════════════════════════════
#  SECTION 10: TRAINING (T4 optimized)
# ══════════════════════════════════════════════════════════════

def train(cfg, csv_path, npz_path):
    """Full training pipeline."""
    dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Device: {dev}")

    torch.manual_seed(cfg.seed)
    np.random.seed(cfg.seed)
    random.seed(cfg.seed)
    os.makedirs(cfg.ckpt_dir, exist_ok=True)
    os.makedirs(cfg.log_dir, exist_ok=True)

    # Build vocab
    df = pd.read_csv(csv_path)
    print(f"CSV: {len(df)} rows")
    vocab, i2t, meta = build_vocab(df, cfg.expr_col)

    # Auto-detect max_vars
    vt = [t for t in vocab if _is_var(t)]
    if vt:
        cfg.max_vars = max(
            int(t.lower().replace("_", "")[1:]) for t in vt
        )
        print(f"max_vars: {cfg.max_vars}")

    # Load data
    vi = get_valid_indices(npz_path)
    ap, al, ki = load_into_ram(npz_path, vi, cfg.max_vars)
    i2p = {r: p for p, r in enumerate(ki)}

    # Match equations to data
    fp = []
    fl = []
    ft = []
    for ri in ki:
        if ri >= len(df):
            continue

        expr_str = str(df[cfg.expr_col].iloc[ri])
        ids = tokenize(expr_str, vocab)

        if ids is None:
            continue
        if len(ids) > cfg.max_expr_len:
            continue
        if not valid_prefix(ids, vocab, meta):
            continue

        fp.append(ap[i2p[ri]])
        fl.append(al[i2p[ri]])
        ft.append(ids)

    print(f"Matched: {len(ft)} equations")
    if not ft:
        raise ValueError("No valid equations!")

    fp = np.array(fp)
    fl = np.array(fl)

    # Precompute stats
    all_sf = precompute_stats(fp, fl, cfg.max_vars)

    # Pre-pad all tokens
    pad_id = vocab["<PAD>"]
    max_len = cfg.max_expr_len
    tok_padded = np.full((len(ft), max_len), pad_id, dtype=np.int64)
    for i, ids in enumerate(ft):
        tok_padded[i, :len(ids)] = ids

    # Convert to tensors
    fp_t = torch.from_numpy(fp)
    fl_t = torch.from_numpy(fl)
    sf_t = torch.from_numpy(all_sf)
    tk_t = torch.from_numpy(tok_padded)

    # Train/val split
    n = len(ft)
    perm = torch.randperm(n)
    sp = int(n * 0.9)
    tr_i = perm[:sp]
    vl_i = perm[sp:]

    tr_dl = DataLoader(
        SRDataset(fp_t[tr_i], fl_t[tr_i], sf_t[tr_i], tk_t[tr_i], cfg, augment=True),
        batch_size=cfg.batch_size,
        shuffle=True,
        num_workers=4,
        pin_memory=True,
        drop_last=True,
        persistent_workers=True,
    )
    vl_dl = DataLoader(
        SRDataset(fp_t[vl_i], fl_t[vl_i], sf_t[vl_i], tk_t[vl_i], cfg, augment=False),
        batch_size=cfg.batch_size,
        shuffle=False,
        num_workers=4,
        pin_memory=True,
        persistent_workers=True,
    )

    # Model
    model = SymbolicRegressionModel(cfg, vocab, meta).to(dev)
    print(f"Parameters: {model.n_params():,}")

    # Optimizer and scheduler
    opt = torch.optim.AdamW(
        model.parameters(),
        lr=cfg.lr,
        weight_decay=cfg.weight_decay,
    )
    total_steps = len(tr_dl) * cfg.epochs

    def lr_fn(s):
        if s < cfg.warmup:
            return s / max(1, cfg.warmup)
        progress = (s - cfg.warmup) / max(1, total_steps - cfg.warmup)
        return 0.5 * (1 + math.cos(math.pi * progress))

    sched = torch.optim.lr_scheduler.LambdaLR(opt, lr_fn)
    loss_fn = nn.CrossEntropyLoss(ignore_index=pad_id, label_smoothing=0.1)
    scaler = torch.cuda.amp.GradScaler(enabled=cfg.amp)

    best = float("inf")
    wait = 0
    hist = []

    print(f"\n{'Ep':>4} | {'TrL':>7} | {'VlL':>7} | {'TrA':>6} | {'VlA':>6} | {'s':>4}")
    print("─" * 42)

    for ep in range(1, cfg.epochs + 1):
        t0 = time.time()

        # ── TRAIN ──
        model.train()
        tl = 0
        tc = 0
        tt = 0

        for dp, pm, sf, tg, _ in tr_dl:
            dp = dp.to(dev, non_blocking=True)
            pm = pm.to(dev, non_blocking=True)
            sf = sf.to(dev, non_blocking=True)
            tg = tg.to(dev, non_blocking=True)

            with torch.cuda.amp.autocast(enabled=cfg.amp):
                lo = model(dp, pm, sf, tg[:, :-1], grammar=False)
                loss = loss_fn(
                    lo.reshape(-1, lo.size(-1)),
                    tg[:, 1:].reshape(-1),
                )

            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
            scaler.step(opt)
            scaler.update()
            opt.zero_grad()
            sched.step()

            with torch.no_grad():
                p = lo.argmax(-1)
                m = tg[:, 1:] != pad_id
                tc += ((p == tg[:, 1:]) & m).sum().item()
                tt += m.sum().item()

            tl += loss.item()

        tl = tl / max(len(tr_dl), 1)
        ta = tc / max(tt, 1)

        # ── VAL ──
        model.eval()
        vl_ = 0
        vc = 0
        vt_ = 0

        with torch.no_grad():
            for dp, pm, sf, tg, _ in vl_dl:
                dp = dp.to(dev, non_blocking=True)
                pm = pm.to(dev, non_blocking=True)
                sf = sf.to(dev, non_blocking=True)
                tg = tg.to(dev, non_blocking=True)

                with torch.cuda.amp.autocast(enabled=cfg.amp):
                    lo = model(dp, pm, sf, tg[:, :-1], grammar=False)
                    loss = loss_fn(
                        lo.reshape(-1, lo.size(-1)),
                        tg[:, 1:].reshape(-1),
                    )

                p = lo.argmax(-1)
                m = tg[:, 1:] != pad_id
                vc += ((p == tg[:, 1:]) & m).sum().item()
                vt_ += m.sum().item()
                vl_ += loss.item()

        vl_ = vl_ / max(len(vl_dl), 1)
        va = vc / max(vt_, 1)
        dt = time.time() - t0

        hist.append({
            "ep": ep,
            "tl": tl,
            "vl": vl_,
            "ta": ta,
            "va": va,
        })

        line = f"{ep:>4} | {tl:>7.4f} | {vl_:>7.4f} | {ta:>5.1%} | {va:>5.1%} | {dt:>3.0f}"

        if vl_ < best:
            best = vl_
            wait = 0
            torch.save(
                {
                    "ep": ep,
                    "model": model.state_dict(),
                    "cfg": cfg,
                    "vocab": vocab,
                    "i2t": i2t,
                    "meta": meta,
                    "vl": vl_,
                    "va": va,
                },
                f"{cfg.ckpt_dir}/best.pt",
            )
            print(line + " ✓")
        else:
            wait += 1
            print(line + f" w={wait}")

        if ep % 10 == 0:
            with open(f"{cfg.log_dir}/hist.json", "w") as f:
                json.dump(hist, f)

        if wait >= cfg.patience:
            print(f"\nEarly stop ep {ep}")
            break

    print(f"\nBest val loss: {best:.4f}")
    return model, vocab, i2t, meta


# ══════════════════════════════════════════════════════════════
#  SECTION 11: INFERENCE
# ══════════════════════════════════════════════════════════════

def load_trained(path, device):
    """Load a trained model from checkpoint."""
    ck = torch.load(path, map_location=device, weights_only=False)
    cfg = ck["cfg"]
    vocab = ck["vocab"]
    i2t = ck["i2t"]
    meta = ck["meta"]

    model = SymbolicRegressionModel(cfg, vocab, meta).to(device)
    model.load_state_dict(ck["model"])
    model.eval()

    print(f"Loaded ep {ck['ep']}, vl={ck['vl']:.4f}, vocab={len(vocab)}")
    return model, vocab, i2t, meta, cfg


def predict_single(model, X, y, vocab, i2t, meta, cfg, device):
    """One-shot prediction: beam search + BFGS."""
    mv = cfg.max_vars

    # Pad X to max_vars
    if X.shape[1] < mv:
        pad = np.zeros((X.shape[0], mv - X.shape[1]))
        X = np.hstack([X, pad])

    # Prepare input
    dp = np.column_stack([X, y]).astype(np.float32)
    sf = compute_stats(dp, len(dp), mv)

    # Normalize
    mu = dp.mean(0, keepdims=True)
    sig = dp.std(0, keepdims=True) + 1e-8
    dn = ((dp - mu) / sig).astype(np.float32)
    dn = np.clip(dn, -10.0, 10.0)

    # Pad to 500
    n = dn.shape[0]
    if n < 500:
        pad = np.zeros((500 - n, dn.shape[1]), dtype=np.float32)
        dn = np.vstack([dn, pad])
        mk = np.zeros(500, dtype=np.float32)
        mk[:n] = 1.0
    else:
        dn = dn[:500]
        mk = np.ones(500, dtype=np.float32)

    # Beam search
    dp_t = torch.tensor(dn).unsqueeze(0)
    mk_t = torch.tensor(mk).unsqueeze(0)
    sf_t = torch.tensor(sf).unsqueeze(0)
    cands = beam_search(model, dp_t, mk_t, sf_t, vocab, meta, cfg, device)

    # Pick best via BFGS
    be = None
    bm = float("inf")
    for ti, lp in cands:
        strs = []
        for i in ti:
            if i in i2t and i2t[i] not in {"<PAD>", "<SOS>", "<EOS>"}:
                strs.append(i2t[i])
        if not strs:
            continue

        e, m = bfgs_fit(strs, X, y, cfg, meta["ar_map"])
        if m < bm:
            bm = m
            be = e

    # R² score
    ss_tot = np.sum((y - y.mean()) ** 2)
    if be and ss_tot > 1e-10:
        r2 = 1 - bm * len(y) / ss_tot
    else:
        r2 = 0.0

    return {"expr": be, "mse": bm, "r2": r2}


# ══════════════════════════════════════════════════════════════
#  RUN
# ══════════════════════════════════════════════════════════════

cfg = Config()

# Bigger model for better accuracy
cfg.d_model = 256
cfg.n_heads = 8
cfg.dec_layers = 6
cfg.dropout = 0.15
cfg.n_subsample = 300
cfg.batch_size = 64
cfg.lr = 1e-4
cfg.warmup = 2000
cfg.epochs = 150
cfg.patience = 20
cfg.max_expr_len = 60

cfg.ckpt_dir = "/content/drive/MyDrive/checkpoints_v2"

csv = "/content/drive/MyDrive/symbolic_data/18march 50k data/Copy of equations_50k_final_input_18march.csv"
npz = "/content/drive/MyDrive/symbolic_data/18march 50k data/Copy of equations_50k_data.npz"

model, vocab, i2t, meta = train(cfg, csv, npz)

Device: cuda
CSV: 50000 rows
  Inferring arities for 31 tokens...
  Vocab: 37  binary:['ADD', 'DIV', 'MUL', 'POW', 'SUB']  unary:['NEG', 'arccos', 'arcsin', 'arctan', 'cos', 'exp', 'log', 'sin', 'sqrt', 'tan', 'tanh']
  vars:['x1', 'x2', 'x3']
  terms:['<C_0>', '<C_10>', '<C_11>', '<C_12>', '<C_1>', '<C_2>', '<C_3>', '<C_4>', '<C_5>', '<C_6>']...
  Parse verify: 5000/5000
max_vars: 3
Scanning NPZ...
Found 44820 valid / 50000
Loading into RAM...


Loading: 100%|██████████| 44820/44820 [04:49<00:00, 154.65it/s]


  Loaded 43510, shape (43510, 500, 4)
Matched: 43250 equations
Precomputing stats...


Stats: 100%|██████████| 43250/43250 [00:55<00:00, 783.38it/s]


  Dataset: 38925, aug=True
  Dataset: 4325, aug=False
Parameters: 6,892,325

  Ep |     TrL |     VlL |    TrA |    VlA |    s
──────────────────────────────────────────
   1 |  3.6699 |  2.6539 | 15.4% | 24.8% |  40 ✓
   2 |  2.6138 |  2.5286 | 25.3% | 27.1% |  41 ✓
   3 |  2.5350 |  2.4832 | 26.8% | 27.9% |  41 ✓
   4 |  2.4895 |  2.4462 | 27.9% | 29.0% |  41 ✓
   5 |  2.4570 |  2.4201 | 28.9% | 29.9% |  41 ✓
   6 |  2.4355 |  2.4050 | 29.4% | 30.2% |  41 ✓
   7 |  2.4197 |  2.3984 | 29.9% | 30.4% |  40 ✓
   8 |  2.4064 |  2.3862 | 30.3% | 30.8% |  41 ✓
   9 |  2.3959 |  2.3801 | 30.7% | 31.2% |  41 ✓
  10 |  2.3872 |  2.3757 | 31.0% | 31.3% |  41 ✓
  11 |  2.3790 |  2.3716 | 31.2% | 31.4% |  41 ✓
  12 |  2.3714 |  2.3677 | 31.6% | 31.6% |  40 ✓
  13 |  2.3643 |  2.3639 | 31.9% | 31.7% |  41 ✓
  14 |  2.3595 |  2.3643 | 32.1% | 31.7% |  41 w=1
  15 |  2.3524 |  2.3598 | 32.3% | 31.9% |  41 ✓
  16 |  2.3474 |  2.3552 | 32.5% | 32.0% |  41 ✓
  17 |  2.3422 |  2.3511 | 32.7% | 32.2% |  